# Урок 04 - Шаблон проектирования использования инструментов

В этом уроке вы узнаете шаблон проектирования **Использование инструментов** для AI-агентов с использованием Microsoft Agent Framework (Python). Мы рассмотрим:

- Определение функций инструментов с декоратором `@tool` и типизированными параметрами
- Предоставление схем инструментов, чтобы модель знала, что делает каждый инструмент
- Управление выполнением инструментов с помощью `approval_mode`
- Возвращение **структурированного вывода** через модели Pydantic и `response_format`

Сценарий — это **агент по бронированию путешествий**, который может находить направления, проверять доступность и получать информацию о рейсах.


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Определение инструментов с декоратором @tool

Декоратор `@tool` превращает обычную функцию Python в инструмент, который агент может вызвать.
Ключевые моменты:

- **Докстринг** становится описанием инструмента, которое видит модель.
- **Аннотации типов** (включая `Annotated` с описаниями) определяют схему инструмента.
- `approval_mode` контролирует, должен ли пользователь одобрять каждый вызов перед его выполнением.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Создание агента с несколькими инструментами

Передайте все три инструмента клиенту, чтобы модель могла вызывать те из них, которые необходимы для ответа на вопрос пользователя.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Структурированный вывод с инструментами

Установив `response_format` в модель Pydantic, агент вынужден возвращать хорошо типизированный JSON-объект вместо свободного текста. Это полезно, когда последующий код должен программно обрабатывать результат.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Шаблоны одобрения инструментов

Параметр `approval_mode` в `@tool` определяет, требуется ли одобрение пользователя перед выполнением вызовов инструмента:

| Режим | Поведение |
|---|---|
| `"never_require"` | Инструмент запускается автоматически — подтверждение пользователя не требуется. |
| `"always_require"` | Каждый вызов должен быть одобрен пользователем перед выполнением. |

Используйте `"always_require"` для инструментов, которые имеют побочные эффекты (например, бронирование рейса, списание средств с кредитной карты), чтобы человек оставался в процессе.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Сводка

В этом уроке вы научились:

1. **Определять инструменты** с помощью декоратора `@tool` с типизированными параметрами и строками документации, которые служат схемой инструмента.  
2. **Комбинировать несколько инструментов**, чтобы агент мог вызывать их последовательно для ответа на сложные запросы.  
3. **Возвращать структурированный результат**, передавая модель Pydantic в качестве `response_format`.  
4. **Контролировать одобрение инструментов** с помощью `approval_mode`, чтобы сохранять участие человека для чувствительных операций.  

Эти шаблоны формируют основу для создания надежных, готовых к промышленному использованию агентов, которые могут безопасно взаимодействовать с внешними системами.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
